# SHAP versus EDEF

SHAP and EDEF answer different questions.

SHAP explains model predictions.

EDEF explains realized model fit.

This notebook uses the same fitted model and the same evaluation sample to show why those are different objects.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

import edef


## Synthetic example

We construct three features:

- `x1_true_signal`: a genuine signal used by the fitted model and aligned with realized outcomes.
- `x2_spurious_model_signal`: a feature the fitted model uses strongly, but which is not useful for realized outcomes in the evaluation sample.
- `x3_noise`: noise.

This creates a simple case where prediction attribution and fit attribution differ.


In [ ]:
rng = np.random.default_rng(123)

n_train = 400
n_eval = 400

# Training sample
X_train = rng.normal(size=(n_train, 3))

# In training, both x1 and x2 appear useful.
y_train = (
    1.0 * X_train[:, 0]
    + 1.0 * X_train[:, 1]
    + rng.normal(scale=0.5, size=n_train)
)

# Evaluation sample
X_eval = rng.normal(size=(n_eval, 3))

# In evaluation, only x1 remains useful.
# x2 still affects predictions because the model was trained to use it,
# but it does not improve realized fit in this sample.
y_eval = (
    1.0 * X_eval[:, 0]
    + rng.normal(scale=0.5, size=n_eval)
)

feature_names = [
    "x1_true_signal",
    "x2_spurious_model_signal",
    "x3_noise",
]


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print("Fitted coefficients")
for name, coef in zip(feature_names, model.coef_):
    print(f"{name:>28s}: {coef: .4f}")


The fitted model uses both `x1` and `x2`. Prediction attribution methods should therefore view both as important for the model's predictions.


In [ ]:
explainer = edef.LinearExplainer(
    model,
    feature_names=feature_names,
)

result = explainer(X_eval, y_eval)

print(result)


## EDEF feature contributions

EDEF attributes realized squared-error improvement in the evaluation sample.

A positive contribution reduces realized loss. A negative contribution increases realized loss.


In [ ]:
result.to_frame()


In [ ]:
ax = result.plot()
ax.set_title("EDEF contributions to realized fit")
plt.show()


In this example, the spurious model signal can receive large prediction attribution but little or negative fit attribution.

That is the central distinction:

- Prediction attribution asks: how did the feature affect the prediction?
- Fit attribution asks: how did the feature affect realized predictive performance?


## SHAP values for the same fitted model

The following cells compute SHAP values for the same model and evaluation sample.

SHAP explains prediction variation, so it should emphasize features the model uses to move predictions.


In [ ]:
try:
    import shap
except ImportError as exc:
    raise ImportError(
        "This notebook requires shap for the SHAP comparison section. "
        "Install with: pip install shap"
    ) from exc


In [ ]:
shap_explainer = shap.LinearExplainer(model, X_train)
shap_values = shap_explainer(X_eval)

shap_importance = np.mean(np.abs(shap_values.values), axis=0)

comparison = pd.DataFrame(
    {
        "feature": feature_names,
        "mean_abs_shap": shap_importance,
        "edef": result.values,
        "edef_se": result.standard_errors,
        "edef_t": result.t_values,
        "edef_share": result.proportions,
    }
).sort_values("mean_abs_shap", ascending=False)

comparison


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

order = np.argsort(shap_importance)
ax.barh(np.asarray(feature_names)[order], shap_importance[order])
ax.set_xlabel("Mean absolute SHAP value")
ax.set_title("SHAP prediction attribution")

plt.show()


The SHAP bar chart measures average prediction attribution magnitude. It does not know the realized outcomes `y_eval`, so it cannot distinguish useful prediction movement from harmful or irrelevant prediction movement in this evaluation sample.


## SHAP plotting compatibility for EDEF

EDEF can export observation-level fit contributions into a `shap.Explanation` object.

This is a plotting adapter, not a semantic conversion: the values remain EDEF fit contributions.


In [ ]:
edef_shap_exp = result.to_shap_explanation(data=X_eval)

shap.plots.beeswarm(edef_shap_exp)


The beeswarm plot above uses SHAP's plotting machinery, but the plotted values are EDEF observation-level contributions to realized fit.

Interpret the horizontal axis as contribution to fit, not contribution to prediction.


## Grouped comparison

Grouping is useful when expanded features map back to original variables or conceptual blocks.


In [ ]:
grouped = result.group(
    ["stable_signal", "spurious_signal", "noise"]
)

print(grouped)


In [ ]:
ax = grouped.plot()
ax.set_title("Grouped EDEF contributions")
plt.show()


## Takeaway

SHAP and EDEF are complementary.

SHAP explains how the model forms predictions.

EDEF explains how features contribute to realized predictive performance.

Because EDEF works observation by observation, it also provides standard errors and t-statistics for feature contributions.
